# Analiza Biznesowa Rynku Nieruchomości - SQL
**Cel:** Wykorzystanie SQL do wyciągnięcia wniosków biznesowych z danych po predykcji.

In [45]:
import sqlite3
import pandas as pd

db_name = 'nieruchomosci.db'
csv_file = 'apartments_final_with_predictions.csv'

# Wczytanie pliku wygenerowanego przez model ML
try:
    df = pd.read_csv(csv_file)
except FileNotFoundError:
    print("Błąd: Nie znaleziono pliku CSV. Uruchom najpierw notatnik z modelem ML, aby wygenerować dane.")

# Połączenie do bazy SQLite
Dodatkowo stworzyłem funkcję pomocniczą do wyświetlania wyników zapytań SQL

In [46]:
# połączenie do bazy SQLite
conn = sqlite3.connect(db_name)

# Zrzucamy dane do tabeli SQL (ETL w wersji mini)
df.to_sql('apartments', conn, if_exists='replace', index=False)

print(f"Baza danych '{db_name}' została utworzona/zaktualizowana.")
print(f"Załadowano {len(df)} wierszy do tabeli 'apartments'.")

# FUNKCJA POMOCNICZA
def run_query(query, title="Wynik zapytania"):
    print(f"\n {title} ")
    return pd.read_sql_query(query, conn)

Baza danych 'nieruchomosci.db' została utworzona/zaktualizowana.
Załadowano 135129 wierszy do tabeli 'apartments'.


# ANALIZA 1: Ranking Miast (Agregacja)
Jak kształtują się średnie ceny w poszczególnych miastach i jak duży jest rynek?

In [47]:
query_1 = """
SELECT 
    city AS Miasto,
    COUNT(*) AS Liczba_Ofert,
    ROUND(AVG(price), 0) AS Srednia_Cena_Calkowita,
    ROUND(AVG(price_per_m2), 0) AS Srednia_Cena_M2
FROM apartments
GROUP BY city
ORDER BY Srednia_Cena_M2 DESC;
"""
display(run_query(query_1, "Ranking cenowy miast"))


 Ranking cenowy miast 


,Miasto,Liczba_Ofert,Srednia_Cena_Calkowita,Srednia_Cena_M2
0,warszawa,45646,1034342.0,17375.0
1,krakow,19720,920192.0,16075.0
2,gdansk,13504,852545.0,14637.0
3,wroclaw,14613,744099.0,13077.0
4,gdynia,5913,827637.0,12877.0
5,poznan,4555,661035.0,10927.0
6,rzeszow,1169,570783.0,10126.0
7,lublin,4146,558342.0,9285.0
8,bialystok,1871,486782.0,9190.0
9,szczecin,4523,558516.0,8918.0


# ANALIZA 2: Top Okazje Inwestycyjne
Sprawdzamy, które miasta oferują najwyższy zwrot z inwestycji, eliminując duplikaty ofert.`zysk > 15%`

In [48]:
query_2 = """
SELECT 
    city,
    squareMeters,
    price AS Cena_Ofertowa,
    ROUND(Predicted_Price, 0) AS Wycena_Modelu,
    ROUND(Predicted_Price - price, 0) AS Potencjalny_Zysk_PLN,
    ROUND(((Predicted_Price - price) / Predicted_Price) * 100, 1) AS Zysk_Procent
FROM apartments
WHERE opportunity_score > 0.15
ORDER BY Potencjalny_Zysk_PLN DESC
LIMIT 10;
"""
display(run_query(query_2, "Największe okazje inwestycyjne"))


 Największe okazje inwestycyjne 


,city,squareMeters,Cena_Ofertowa,Wycena_Modelu,Potencjalny_Zysk_PLN,Zysk_Procent
0,warszawa,140.86,1500000,2737918.0,1237918.0,45.2
1,warszawa,105.00,1200000,2180136.0,980136.0,45.0
2,warszawa,138.79,1100000,1990435.0,890435.0,44.7
3,krakow,113.00,1080000,1875867.0,795867.0,42.4
4,krakow,80.10,799000,1585416.0,786416.0,49.6
5,warszawa,104.10,1435000,2152349.0,717349.0,33.3
6,warszawa,84.80,845000,1491077.0,646077.0,43.3
7,gdansk,67.50,950000,1586119.0,636119.0,40.1
8,warszawa,131.20,940000,1568843.0,628843.0,40.1
9,gdansk,84.06,975000,1595255.0,620255.0,38.9


# ANALIZA 3: Ranking z Window Functions
**Ranking najdroższych mieszkań** w ramach każdego miasta.

In [49]:
query_3 = """
WITH RankingMieszkan AS (
    SELECT 
        city,
        price,
        squareMeters,
        RANK() OVER (PARTITION BY city ORDER BY price DESC) as ranking_w_miescie
    FROM apartments
)
SELECT DISTINCT * 
FROM RankingMieszkan
WHERE ranking_w_miescie <= 3;
"""
display(run_query(query_3, "Top 3 najdroższe mieszkania w każdym mieście (Window Function)"))


 Top 3 najdroższe mieszkania w każdym mieście (Window Function) 


,city,price,squareMeters,ranking_w_miescie
0,bialystok,905000,75.44,1
1,bialystok,899000,60.00,2
2,bialystok,875000,68.32,3
3,bydgoszcz,1000000,124.50,1
4,bydgoszcz,1000000,91.00,1
5,czestochowa,795000,74.46,1
6,gdansk,2499000,95.70,1
7,gdansk,2499000,120.00,1
8,gdansk,2426922,120.10,3
9,gdynia,3055400,146.61,1


# ANALIZA 4: Segmentacja Rynku
**Struktura cenowa rynku ->** Ile jest ofert budżetowych, a ile premium?

In [50]:
query_4 = """
SELECT 
    city,
    CASE 
        WHEN price_per_m2 < 9000 THEN '1. Budżetowe (<9k/m2)'
        WHEN price_per_m2 BETWEEN 9000 AND 14000 THEN '2. Standard (9k-14k/m2)'
        ELSE '3. Premium (>14k/m2)'
    END AS Segment_Cenowy,
    COUNT(*) AS Liczba_Ofert,
    ROUND(AVG(price), 0) AS Srednia_Cena_Calkowita
FROM apartments
WHERE city IN ('warszawa', 'krakow', 'wroclaw', 'poznan', 'gdansk', 'lodz')
GROUP BY city, Segment_Cenowy
ORDER BY city, Segment_Cenowy;
"""
display(run_query(query_4, "Segmentacja rynku w dużych miastach"))


 Segmentacja rynku w dużych miastach 


,city,Segment_Cenowy,Liczba_Ofert,Srednia_Cena_Calkowita
0,gdansk,1. Budżetowe (<9k/m2),281,662713.0
1,gdansk,2. Standard (9k-14k/m2),7056,716889.0
2,gdansk,3. Premium (>14k/m2),6167,1016406.0
3,krakow,1. Budżetowe (<9k/m2),210,627619.0
4,krakow,2. Standard (9k-14k/m2),6560,795322.0
5,krakow,3. Premium (>14k/m2),12950,988191.0
6,lodz,1. Budżetowe (<9k/m2),6004,421373.0
7,lodz,2. Standard (9k-14k/m2),2528,536474.0
8,poznan,1. Budżetowe (<9k/m2),912,631423.0
9,poznan,2. Standard (9k-14k/m2),3181,644955.0


# ANALIZA 5: Wiarygodność Statystyczna
Średnie ceny w miastach, gdzie mamy duży sample.

In [51]:
query_5 = """
SELECT 
    city,
    COUNT(*) as Probka_Danych,
    ROUND(AVG(price_per_m2), 0) as Srednia_Cena_m2
FROM apartments
GROUP BY city
HAVING COUNT(*) > 50  -- Tylko miasta z min. 50 ofertami
ORDER BY Srednia_Cena_m2 DESC
LIMIT 10;
"""
display(run_query(query_5, "Najdroższe miasta (tylko istotne statystycznie)"))


 Najdroższe miasta (tylko istotne statystycznie) 


,city,Probka_Danych,Srednia_Cena_m2
0,warszawa,45646,17375.0
1,krakow,19720,16075.0
2,gdansk,13504,14637.0
3,wroclaw,14613,13077.0
4,gdynia,5913,12877.0
5,poznan,4555,10927.0
6,rzeszow,1169,10126.0
7,lublin,4146,9285.0
8,bialystok,1871,9190.0
9,szczecin,4523,8918.0


# Zamknięcie połączenia z bazą danych

In [52]:
conn.close()